# 02 · Qwen을 Jev 호환 결정 모델로 학습
**대상:** 딥러닝 경험자 · **환경:** Linux, Python 3.11/3.12, 단일 NVIDIA CUDA GPU.
**권장 시작 환경:** VRAM 16GB 이상, 디스크 30GB 이상 여유. 이는 실습 계획값이며 실제 사용량은 아래에서 측정한다.
Jev 자체의 비공개 가중치를 학습하는 것이 아니다. Unsloth가 공개 Qwen 본체에 결정 헤드를 붙인다.

순서: 환경 → 원본 행 분할 → LoRA/헤드 학습 → calibration → 독립 test → 저장 → 재로딩 → 예측 비교.
기본값은 0.8B / 60 steps의 동작 확인용이다. 공식 벤치마크 재현 실험이 아니다.
이 파일 하나만 서버로 옮겨도 실행된다. 결과는 현재 작업 폴더의 `artifacts/` 아래에 저장된다.

이 자료 작성 환경에는 NVIDIA GPU가 없어 **실제 학습은 미실행**이다. API와 셀 구문은 공식 소스에 대조했다.


## 1. 설치 (필요할 때 한 번)
먼저 [PyTorch 공식 선택기](https://pytorch.org/get-started/locally/)에 맞는 CUDA PyTorch를 설치한다.
아래 패키지 버전은 2026-10-13에 확인한 배포 버전이며 GPU 전체 의존성 lock은 아니다.
새 환경에서 `INSTALL=True`로 실행한 후 **커널을 재시작하고 False로 복원**한다.
Unsloth는 transformers/peft보다 먼저 import해야 한다. 기존 환경을 덮어쓰기보다 전용 환경을 사용한다.


In [ ]:
INSTALL = False
if INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'unsloth==2026.10.4', 'unsloth_zoo==2026.10.4',
                           'datasets>=3,<5', 'huggingface_hub>=0.34,<2'])
    print('설치 완료. 커널을 재시작한 뒤 INSTALL=False로 실행하세요.')


In [ ]:
# transformers / peft / torch보다 먼저 Unsloth를 import한다.
import importlib.util
if importlib.util.find_spec('unsloth') is None:
    raise RuntimeError('Unsloth가 없습니다. 설치 셀 또는 requirements-gpu.txt를 사용하세요.')
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
import torch
import inspect, json, os, random, platform, subprocess, sys, time, hashlib, gc
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import TrainingArguments, set_seed

if not torch.cuda.is_available():
    raise RuntimeError('이 학습 노트북은 NVIDIA CUDA GPU가 필요합니다. CPU 실습은 01을 사용하세요.')
required = ['build_dataset', 'evaluate', 'calibrate', 'predict', 'for_training']
assert all(hasattr(FastDecisionModel, name) for name in required), '결정 모델 API가 없는 Unsloth 버전'
assert 'processing_class' in inspect.signature(DecisionTrainer.__init__).parameters or any(
    p.kind == p.VAR_KEYWORD for p in inspect.signature(DecisionTrainer.__init__).parameters.values())
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('torch / CUDA:', torch.__version__, torch.version.cuda)
print('Unsloth:', version('unsloth'))


## 2. 실행 설정과 재현 정보
`SMOKE=True`는 입력 240행, 테스트 40행, 60 steps. `False`는 전체 train/test와 2 epochs.
기본 본체는 0.8B이며 `MODEL_ID`를 4B로 바꿀 수 있다. 메모리는 batch=1부터 확인한다.
여러 GPU가 보이더라도 이 실습은 단일 GPU 기준이다. 서버에서 `CUDA_VISIBLE_DEVICES=0`으로 커널을 시작한다.
재현 시 아래 revision 값을 이전 `run_manifest.json`의 SHA로 채운다.


In [ ]:
SEED = 3407
SMOKE = True
MODEL_ID = 'unsloth/Qwen3.5-0.8B'
MODEL_REVISION = None  # 이전 manifest의 model_revision SHA를 넣으면 같은 가중치를 사용
DATASET_ID = 'LocalLLaMA/typed-decisions'
DATASET_REVISION = None
MAX_SEQ_LENGTH = 2048
set_seed(SEED)
api = HfApi()
model_revision = MODEL_REVISION or api.model_info(MODEL_ID).sha
resolved_dataset_revision = DATASET_REVISION or api.dataset_info(DATASET_ID).sha
RUN_DIR = Path('artifacts') / ('qwen-decisions-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN_DIR.mkdir(parents=True, exist_ok=False)
manifest = dict(seed=SEED, smoke=SMOKE, model=MODEL_ID, model_revision=model_revision,
                dataset=DATASET_ID, dataset_revision=resolved_dataset_revision,
                max_seq_length=MAX_SEQ_LENGTH, python=platform.python_version(),
                gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,
                versions={p: version(p) for p in ['torch','unsloth','unsloth_zoo','transformers','datasets','peft']})
(RUN_DIR / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print('outputs:', RUN_DIR.resolve())


## 3. 데이터 로드와 누수 검사
데이터는 `state`, `questions`, `gold`를 포함한다. JSON 문자열인 필드를 파싱한다.
`gold`, `factors`, `label_agreement`는 모델 입력에 넣지 않는다.
동일 state의 모든 질문은 같은 분할에 둔다. exact-state 중복을 제거하지만 의미상 유사 문장까지 검출하지는 않는다.
train/test 사이 state가 중복되면 학습을 중단한다. 커스텀 데이터는 고객/문서/시간 그룹 단위 분할도 필요하다.


In [ ]:
def parsed(value):
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value

def state_key(row):
    canonical = json.dumps(parsed(row['state']), ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(canonical.encode()).hexdigest()

def unique_rows(rows):
    found = {}
    for row in rows:
        found.setdefault(state_key(row), row)
    return list(found.values())

raw_train = load_dataset(DATASET_ID, 'all', split='train', revision=resolved_dataset_revision)
raw_test = load_dataset(DATASET_ID, 'all', split='test', revision=resolved_dataset_revision)
train_rows, test_rows = unique_rows(raw_train), unique_rows(raw_test)
assert not ({state_key(r) for r in train_rows} & {state_key(r) for r in test_rows}), 'train/test state leakage'
print('deduplicated:', len(raw_train)-len(train_rows), len(raw_test)-len(test_rows))
rng = random.Random(SEED)
rng.shuffle(train_rows)
rng.shuffle(test_rows)
if SMOKE:
    train_rows, test_rows = train_rows[:240], test_rows[:40]
assert len(train_rows) >= 30 and test_rows, '분할하기에 데이터가 부족합니다.'
a, b = int(len(train_rows)*.7), int(len(train_rows)*.85)
raw_splits = {'train': train_rows[:a], 'validation': train_rows[a:b],
              'calibration': train_rows[b:], 'test': test_rows}
keys = {name: {state_key(r) for r in rows} for name, rows in raw_splits.items()}
for i, name in enumerate(keys):
    for other in list(keys)[i+1:]:
        assert keys[name].isdisjoint(keys[other]), (name, other)
manifest['split_state_hashes'] = {name: sorted(values) for name, values in keys.items()}
print({name: len(rows) for name, rows in raw_splits.items()})
sample = raw_splits['train'][0]
print('state preview:', str(parsed(sample['state']))[:250])
print('questions:', json.dumps(parsed(sample['questions']), ensure_ascii=False, indent=2)[:1500])
print('gold:', str(parsed(sample['gold']))[:500])


## 4. 모델과 LoRA
4-bit는 본체 저장/계산 메모리를 줄인다. LoRA는 본체의 업데이트를 학습하고 새 결정 헤드도 학습한다.
새 헤드이므로 아래 학습 전 수치는 **무학습 결정 헤드 baseline**이다. Qwen의 채팅/JSON zero-shot 성능이 아니다.
실제 trainable parameter 수와 메모리를 출력해 가정을 확인한다.


In [ ]:
model, tokenizer = FastDecisionModel.from_pretrained(
    model_name=MODEL_ID, revision=model_revision,
    max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
)
model = FastDecisionModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    use_gradient_checkpointing='unsloth', random_state=SEED,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print({'trainable_parameter_elements': trainable, 'all_parameter_elements': total})
print('4-bit 파라미터의 저장 표현 때문에 위 수는 원본 모델의 정확한 parameter count와 다를 수 있습니다.')


## 5. 토큰화와 변환 보고서
Unsloth 변환기는 잘못된 질문을 건너뛸 수 있다. 조용히 성능 비교에서 빠지지 않게 skipped가 있으면 중단한다.
truncated는 별도로 기록한다. 길이에 걸린 state 끝의 정답 근거가 사라졌는지 확인한다.
행 수와 질문 수는 다르다. Clef식 항목은 한 행에 여러 질문을 묶을 수 있다.


In [ ]:
items, reports = {}, {}
for name, rows in raw_splits.items():
    items[name], reports[name] = FastDecisionModel.build_dataset(rows, tokenizer, model)
    print(name, 'items=', len(items[name]), 'report=', reports[name])
    assert items[name], f'{name}: no usable items'
    assert reports[name]['skipped'] == 0, f'{name}: inspect skipped questions before continuing'
manifest['build_reports'] = reports
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str))
FastDecisionModel.for_inference(model)
before = FastDecisionModel.evaluate(model, tokenizer, items['validation'], batch_size=1)
print('validation / random head:', before)


## 6. 실제 학습
본체 LoRA LR=2e-4, 헤드 LR=1e-4를 명시한다. batch=1, accumulation=8은 단일 GPU에서 유효 batch 8.
validation으로 진행을 관찰하고 calibration/test는 학습에 넣지 않는다.
OOM이면 길이·batch·모델 크기를 낮추고, 변경 내용을 manifest에 기록한다.
아래 셀은 실제 GPU 계산을 시작한다.


In [ ]:
FastDecisionModel.for_training(model)
args = TrainingArguments(
    output_dir=str(RUN_DIR / 'checkpoints'),
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    max_steps=60 if SMOKE else -1, num_train_epochs=2,
    warmup_ratio=0.1, learning_rate=2e-4, lr_scheduler_type='cosine', weight_decay=0.01,
    bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
    eval_strategy='steps', eval_steps=20, logging_steps=10,
    save_strategy='steps', save_steps=20, save_total_limit=2,
    report_to='none', seed=SEED, data_seed=SEED,
)
trainer = DecisionTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=items['train'], eval_dataset=items['validation'],
    head_learning_rate=1e-4, args=args,
)
manifest['training_args'] = args.to_dict()
manifest['head_learning_rate'] = 1e-4
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str))
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()
print('elapsed seconds:', round(time.perf_counter() - started, 1))
print('peak reserved GiB:', round(torch.cuda.max_memory_reserved() / 2**30, 2))
print(train_result.metrics)


## 7. 학습 전후 → 보정 → 최종 test
보정 함수가 calibration split에서 추정한 온도를 모델 설정에 저장한다.
공식 evaluate 지표의 이름/정의는 설치한 버전을 따른다. 결과 dict를 그대로 저장한다.
최종 test는 이 단계에서만 읽고, test 결과를 보고 설정을 고르면 별도 최종 데이터가 다시 필요하다.


In [ ]:
FastDecisionModel.for_inference(model)
after = FastDecisionModel.evaluate(model, tokenizer, items['validation'], batch_size=1)
calibration = FastDecisionModel.calibrate(model, tokenizer, items['calibration'], batch_size=1)
calibrated_validation = FastDecisionModel.evaluate(model, tokenizer, items['validation'], batch_size=1)
final_test = FastDecisionModel.evaluate(model, tokenizer, items['test'], batch_size=1)
results = {'validation_before_random_head': before, 'validation_after_training': after,
           'calibration_fit_report': calibration, 'validation_after_calibration': calibrated_validation,
           'test_after_calibration': final_test, 'train': train_result.metrics}
print(json.dumps(results, indent=2, default=str))
(RUN_DIR / 'metrics.json').write_text(json.dumps(results, indent=2, default=str))


## 8. 새 입력 추론과 저장
한국어 예시는 정량 평가가 아닌 전이 동작 관찰이다. 영어 합성 학습 데이터에서의 성능이 한국어로 보장되지는 않는다.
어댑터 저장에는 결정 헤드도 포함된다. tokenizer는 Unsloth의 결정 모델 저장 경로가 처리한다.
merged export는 16-bit 모델로 커지므로 필요한 경우에만 실행한다.


In [ ]:
questions = {
    'team': {'type':'choice', 'instructions':'담당 팀을 선택하세요.',
             'criteria':{'billing':'결제 및 환불', 'technical':'장애 및 오류', 'sales':'가격 및 계약'}},
    'refund': {'type':'noul', 'instructions':'고객이 환불을 요청합니까?'},
    'urgency': {'type':'score', 'instructions':'처리 긴급도', 'criteria':['여유 있음','곧 처리','오늘 처리']},
}
state = '어제 결제가 두 번 되었습니다. 오늘 중복 결제 금액을 환불해 주세요.'
prediction_before_save = FastDecisionModel.predict(model, tokenizer, state, questions)
print(json.dumps(prediction_before_save, ensure_ascii=False, indent=2))
ADAPTER_DIR = RUN_DIR / 'adapter'
model.save_pretrained(str(ADAPTER_DIR))
EXPORT_MERGED = False
if EXPORT_MERGED:
    model.save_pretrained_merged(str(RUN_DIR / 'merged'))
print('saved:', ADAPTER_DIR.resolve())


## 9. GPU 메모리 해제 후 재로딩 검증
두 모델을 동시에 GPU에 두지 않는다. 어댑터 로딩은 원본 본체를 다시 가져오므로 원본 모델 접근이 필요하다.
양자화/직렬화로 작은 차이가 생길 수 있다. 확률 최대 차이와 선택 결과를 출력하고 허용오차를 검사한다.
실패하면 통과로 간주하지 말고 온도 저장, revision, dtype과 패키지 버전을 확인한다.


In [ ]:
del trainer, model, tokenizer
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = FastDecisionModel.from_pretrained(
    model_name=str(ADAPTER_DIR), max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
)
FastDecisionModel.for_inference(model)
prediction_after_load = FastDecisionModel.predict(model, tokenizer, state, questions)
max_delta = 0.0
for name, previous in prediction_before_save.items():
    current = prediction_after_load[name]
    assert previous['answer'] == current['answer'], f'{name}: answer changed after reload'
    assert set(previous['probabilities']) == set(current['probabilities'])
    max_delta = max(max_delta, max(abs(float(p)-float(current['probabilities'][k]))
                                  for k, p in previous['probabilities'].items()))
print('max probability delta:', max_delta)
assert max_delta < 0.02, '저장/재로딩 확률 차이를 조사하세요.'
(RUN_DIR / 'reload_check.json').write_text(json.dumps({'max_delta':max_delta, 'passed':True}))


## 10. 과제와 실험 설계
1. 같은 데이터/revision/seed에서 60 steps와 2 epochs를 비교한다.
2. 한국어 문의와 정답을 별도 수집해 test를 만든다. 영어 test 결과와 나눠 보고한다.
3. 선택지 순서를 바꾼 요청의 확률을 키로 정렬해 비교한다.
4. 최소 3개 seed와 더 큰 test에서 변동성을 확인한다. 40행 smoke test로 모델 우열을 주장하지 않는다.

**해설 가이드:** 검증 지표로 설정을 선택 → calibration split으로 보정 → 최종 test 한 번 →
accuracy, soft-gold loss, ECE와 샘플 수, GPU/지연시간을 함께 보고한다.

### 로컬 서빙으로 연결
`EXPORT_MERGED=True`로 저장한 뒤 GPU 서버 터미널에서 (실제 절대경로로 변경):
```bash
UNSLOTH_SYSTEMONE_MODEL=/absolute/path/to/merged unsloth studio -H 127.0.0.1 -p 8888
```
Studio Settings → API → Decision API에서 Serve requests를 켠다. 이어서 03을 실행한다.
다른 PC의 노트북에서 접속할 때 `ssh -L 8888:127.0.0.1:8888 user@gpu-server`로 터널을 연다.
기존 Laya 선택과 새 학습 모델 선택은 별도 작업이며 03에서는 실제 응답 model도 기록한다.

출처: [학습 가이드](https://unsloth.ai/docs/new/train-your-own-decision-model-with-unsloth),
[공식 노트북](https://github.com/unslothai/notebooks/blob/26a618143bb36ecb36050b1bcb38b602b64a2ee5/nb/Qwen3_5_(4B)-Decision.ipynb),
[decision.py](https://github.com/unslothai/unsloth/blob/1e7d78415b93e4b9a036f07fdb953de11363e2dd/unsloth/models/decision.py),
[데이터 카드](https://huggingface.co/datasets/LocalLLaMA/typed-decisions).
